# Phase 2 — Data Cleaning for Foodie India MP1

This notebook is the interactive version of `scripts/01_clean_data.py`.

It reads the raw `Restaurant.xlsx`, fixes typos/formatting errors, flags
domain-invalid values as missing, imputes them, and writes two outputs:

- `data/restaurant_clean_with_na.csv` — typos fixed, invalid values set to `NaN`
- `data/restaurant_clean_final.csv` — the above, plus missing values imputed

Every cleaning decision is documented with its rationale inline below (and in `log.md`).

**Note on paths:** this notebook lives in `notebooks/`, so file paths are written
relative to the project root using `../`.

**Note (2026-09-24):** the team has since adopted a teammate-provided file,
`data/Restaurant_cleaned_final.csv`, as the actual final dataset used
downstream (`02_eda.ipynb` reads that one, not this notebook's output) —
see `log.md`. This notebook is kept running as an independent,
fully-documented cross-check.

## 1. Imports

- `pandas` does the heavy lifting: reading the Excel file, cleaning columns, and writing CSVs.
- `numpy` is only used for `np.nan`, the value we assign to anything we decide is invalid/unrecoverable.

In [1]:
import pandas as pd
import numpy as np

## 2. File paths

Paths are relative to the project root (one level up from this `notebooks/` folder).

In [2]:
RAW_PATH = "../Restaurant.xlsx"
OUT_WITH_NA = "../data/restaurant_clean_with_na.csv"
OUT_FINAL = "../data/restaurant_clean_final.csv"

## 3. Fixing numeric typos (`Amount`, `Tip`)

`Amount` and `Tip` were entered with stray punctuation instead of a decimal
point (e.g. `'28,87'`, `'25-89'`) or trailing junk characters (e.g. `'2.0\``').

A single comma/dash in a currency-like string is almost certainly a decimal
separator typo, not subtraction or a thousands separator, since it appears
exactly once and in the expected decimal position. The function below:

1. Passes numeric values straight through.
2. Replaces a single `,` or a single non-leading `-` with `.`.
3. Strips any remaining non-numeric junk characters.
4. Converts to `float`, falling back to `NaN` if the result still isn't parseable.

In [3]:
def fix_numeric_typo(v):
    """Amount/Tip were entered with stray punctuation instead of a decimal
    point (e.g. '28,87', '25-89') or trailing junk characters (e.g. '2.0`').
    A single comma/dash in a currency-like string is almost certainly a
    decimal separator typo, not subtraction or a thousands separator, since
    it appears exactly once and in the expected decimal position."""
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if "," in s:
        s = s.replace(",", ".", 1)
    elif s.count("-") == 1 and not s.startswith("-"):
        s = s.replace("-", ".", 1)
    s = "".join(ch for ch in s if ch.isdigit() or ch == "." or ch == "-")
    try:
        return float(s)
    except ValueError:
        return np.nan

## 4. Typo maps for categorical columns

`Gender`, `Smoker`, `Day`, and `Time` were typed freehand, so they contain
abbreviations and misspellings. Each dictionary below maps a lowercase,
stripped version of an observed value to its canonical label.

Values that are ambiguous between two categories (e.g. `'San'` could be a
typo for `'Sun'` or `'Sat'`) are deliberately **left out** of the map, so
they fall through to `NaN` rather than being guessed.

In [4]:
GENDER_MAP = {
    "male": "Male", "m": "Male", "mal": "Male", "mle": "Male",
    "female": "Female", "f": "Female", "fe": "Female", "fmle": "Female",
    "femle": "Female", "fem": "Female",
}

SMOKER_MAP = {"no": "No", "n": "No", "yes": "Yes", "y": "Yes"}
# 's' alone doesn't unambiguously match "Yes" or "No" -> left unmapped (becomes NaN)

DAY_MAP = {
    "thur": "Thur", "thurs": "Thur", "trhurs": "Thur", "th": "Thur", "t": "Thur",
    "fri": "Fri", "friday": "Fri",
    "sat": "Sat", "saturday": "Sat",
    "sun": "Sun", "sn": "Sun",
}
# 'S', 'SS', 'SSS', 'San', 'Ft' are equally consistent with more than one day
# (e.g. 'San' is a 2/3-letter match to both "Sun" and "Sat") -> left unmapped

TIME_MAP = {
    "dinner": "Dinner", "diner": "Dinner", "dd": "Dinner", "ddd": "Dinner",
    "di": "Dinner", "din": "Dinner", "d": "Dinner",
    "lunch": "Lunch", "l": "Lunch", "lu": "Lunch", "lan": "Lunch",
}
# 'LD', 'er', 'Afd', 'Din/Lun' don't map confidently to a single value -> NaN

## 5. Helper: apply a typo map

Normalizes a column (string cast, whitespace-stripped, lowercased) and maps
it through the given dictionary. Anything not present in the dictionary
becomes `NaN` — that's `pandas`' default behavior for `.map()` on unmatched keys.

In [5]:
def map_category(series, mapping):
    return series.astype(str).str.strip().str.lower().map(mapping)

## 6. Load the raw data

We only need seven columns for this analysis, so we select them right away
and work on a `.copy()` to avoid `SettingWithCopyWarning`s later.

In [6]:
df = pd.read_excel(RAW_PATH)
df = df[["Amount", "Tip", "Gender", "Smoker", "Day", "Time", "Partysize"]].copy()
n_raw = len(df)

# `report` accumulates human-readable counts of every cleaning decision we make,
# so we can print a summary at the end (and cross-check it against log.md).
report = {}

df.head()

,Amount,Tip,Gender,Smoker,Day,Time,Partysize
0,16.99,1.01,Female,No,Sun,Dinner,2.0
1,10.34,1.66,Male,No,Sun,Dinner,3.0
2,21.01,3.5,Male,No,Sun,Dinner,3.0
3,23.68,3.31,Mal,No,Sun,Dinner,2.0
4,24.59,3.61,Female,No,Sun,Dinner,4.0


## 7. Clean `Amount`

After fixing numeric typos, a **negative bill is physically impossible**,
so any negative `Amount` is treated as invalid and set to `NaN` rather than
guessed at (e.g. by taking its absolute value).

In [7]:
df["Amount"] = df["Amount"].apply(fix_numeric_typo)
invalid_amount = df["Amount"] < 0  # a negative bill is physically impossible
report["Amount: negative (invalid)"] = int(invalid_amount.sum())
df.loc[invalid_amount, "Amount"] = np.nan

report["Amount: negative (invalid)"]

1

## 8. Clean `Tip`

Tips of 288 / 300 / 3487 on ~\$21–22 bills aren't plausible generosity —
they're decimal-placement errors. Since we can't recover the intended value
with confidence, we mark anything over \$100 as invalid rather than guess.

In [8]:
df["Tip"] = df["Tip"].apply(fix_numeric_typo)
# Tips of 288 / 300 / 3487 on ~$21-22 bills are not plausible generosity,
# they're decimal-placement errors; we don't guess the intended value,
# we mark them invalid like any other unrecoverable entry.
invalid_tip = df["Tip"] > 100
report["Tip: implausibly large (invalid)"] = int(invalid_tip.sum())
df.loc[invalid_tip, "Tip"] = np.nan

report["Tip: implausibly large (invalid)"]

3

## 9. Clean `Gender`

Apply the typo map, then record how many values were corrected and how many
remained unresolved (and are therefore now missing).

In [9]:
mapped = map_category(df["Gender"], GENDER_MAP)
report["Gender: typo values corrected"] = int((mapped != df["Gender"]).sum())
df["Gender"] = mapped
report["Gender: unresolved -> missing"] = int(df["Gender"].isna().sum())

df["Gender"].value_counts(dropna=False)

Gender
Male      209
Female    156
Name: count, dtype: int64

## 10. Clean `Smoker`

Same pattern as `Gender`. We compare against non-null mapped values only
(`mapped.notna()`) when counting "corrected" values, since a value that
maps to `NaN` wasn't "corrected" — it's now missing.

In [10]:
mapped = map_category(df["Smoker"], SMOKER_MAP)
report["Smoker: typo values corrected"] = int(
    ((mapped != df["Smoker"]) & mapped.notna()).sum()
)
df["Smoker"] = mapped
report["Smoker: missing/unresolved"] = int(df["Smoker"].isna().sum())

df["Smoker"].value_counts(dropna=False)

Smoker
No     212
Yes    143
NaN     10
Name: count, dtype: int64

## 11. Clean `Day`

Two kinds of rows can't be resolved here: the cell was left blank, or a
value was entered that doesn't map confidently to a single day (e.g. `S`,
`SSS`, `San`, `Ft` — `San` is an equally close match to both "Sun" and
"Sat", a genuine tie). For both cases, rather than impute a guessed day,
the row is **dropped entirely** — there's no reliable signal to fill it
from, unlike `Time`/`Smoker`/`Amount`, which have real group structure to
impute from.

In [11]:
raw_day = df["Day"]
mapped = map_category(df["Day"], DAY_MAP)
report["Day: typo values corrected"] = int(
    ((mapped != raw_day) & mapped.notna()).sum()
)
blank_day = raw_day.isna()
ambiguous_day = mapped.isna() & raw_day.notna()
report["Day: blank (dropped)"] = int(blank_day.sum())
report["Day: ambiguous typo (dropped)"] = int(ambiguous_day.sum())
df["Day"] = mapped
df = df[~(blank_day | ambiguous_day)].reset_index(drop=True)

df["Day"].value_counts(dropna=False)

Day
Sat     115
Sun     108
Thur     90
Fri      39
Name: count, dtype: int64

## 12. Clean `Time`

In [12]:
mapped = map_category(df["Time"], TIME_MAP)
report["Time: typo values corrected"] = int(
    ((mapped != df["Time"]) & mapped.notna()).sum()
)
df["Time"] = mapped
report["Time: missing/unresolved"] = int(df["Time"].isna().sum())

df["Time"].value_counts(dropna=False)

Time
Dinner    218
Lunch     129
NaN         5
Name: count, dtype: int64

## 13. Clean `Partysize`

The observed valid range is 1–6 guests. Anything `<= 0`, `> 6` (e.g. 200,
45, 22), or non-integer (e.g. 0.2 — a party has a whole number of people)
is treated as a data-entry error, not a real party size, and set to `NaN`.

In [13]:
df["Partysize"] = pd.to_numeric(df["Partysize"], errors="coerce")
# Observed valid range is 1-6 guests; anything <=0, >6 (200, 45, 22, ...),
# or non-integer (0.2 -- a party has a whole number of people) is a
# data-entry error rather than a real party size. We do not guess the
# intended digit -> treat as invalid/missing.
non_integer = (df["Partysize"] % 1 != 0) & df["Partysize"].notna()
invalid_party = (df["Partysize"] <= 0) | (df["Partysize"] > 6) | non_integer
report["Partysize: domain-invalid (<=0 or >6)"] = int(invalid_party.sum())
df.loc[invalid_party, "Partysize"] = np.nan
report["Partysize: originally blank"] = int(
    pd.read_excel(RAW_PATH)["Partysize"].isna().sum()
)

report["Partysize: domain-invalid (<=0 or >6)"], report["Partysize: originally blank"]

(14, 3)

## 14. Save the "with NA" checkpoint

At this point every typo is fixed and every domain-invalid value is `NaN`,
but nothing has been imputed yet. This is saved separately so the effect of
imputation (next section) can always be inspected against the pre-imputation
state.

In [14]:
df.to_csv(OUT_WITH_NA, index=False)
print(f"Saved: {OUT_WITH_NA}")

Saved: ../data/restaurant_clean_with_na.csv


## 15. Imputation strategy: group-wise, not a single global constant

A single global mode/median (e.g. "every missing `Smoker` becomes `No`")
ignores that these fields correlate with each other in real dining patterns
(`Thur` is overwhelmingly `Lunch`, `Sat`/`Sun` overwhelmingly `Dinner` — see
`log.md` for the crosstab). Filling from the value typical of a similar
subgroup (same `Day`+`Time`) is a small step up in realism while staying
simple enough to justify in the report, with a global fallback for the rare
case a group has no usable data.

The two helper functions below implement that pattern:
- `impute_categorical_grouped` fills missing categorical values with the
  **mode** of rows sharing the same key columns (falling back to the global
  mode if the group itself has no usable data).
- `impute_numeric_grouped` does the same with the **median** for numeric
  columns.

In [15]:
imputed = df.copy()

def impute_categorical_grouped(col, group_cols):
    if not imputed[col].isna().any():
        return
    missing_idx = imputed[imputed[col].isna()].index
    global_mode = imputed[col].mode(dropna=True)[0]
    filled_from_group, filled_from_fallback = 0, 0
    for idx in missing_idx:
        key = imputed.loc[idx, group_cols]
        if key.isna().any():
            imputed.loc[idx, col] = global_mode
            filled_from_fallback += 1
            continue
        mask = (imputed[group_cols] == key).all(axis=1) & imputed[col].notna()
        group_vals = imputed.loc[mask, col]
        if len(group_vals) == 0:
            imputed.loc[idx, col] = global_mode
            filled_from_fallback += 1
        else:
            imputed.loc[idx, col] = group_vals.mode().iloc[0]
            filled_from_group += 1
    report[f"{col}: imputed via mode within {'+'.join(group_cols)} group"] = filled_from_group
    if filled_from_fallback:
        report[f"{col}: imputed via global mode (group had no data)"] = filled_from_fallback


def impute_numeric_grouped(col, group_cols):
    if not imputed[col].isna().any():
        return
    missing_idx = imputed[imputed[col].isna()].index
    global_median = imputed[col].median()
    filled_from_group, filled_from_fallback = 0, 0
    for idx in missing_idx:
        key = imputed.loc[idx, group_cols]
        if key.isna().any():
            imputed.loc[idx, col] = global_median
            filled_from_fallback += 1
            continue
        mask = (imputed[group_cols] == key).all(axis=1) & imputed[col].notna()
        group_vals = imputed.loc[mask, col]
        if len(group_vals) == 0:
            imputed.loc[idx, col] = global_median
            filled_from_fallback += 1
        else:
            imputed.loc[idx, col] = group_vals.median()
            filled_from_group += 1
    report[f"{col}: imputed via median within {'+'.join(group_cols)} group"] = filled_from_group
    if filled_from_fallback:
        report[f"{col}: imputed via global median (group had no data)"] = filled_from_fallback

## 16. Run imputation, in a deliberate order

`Day` has no missing/unresolved values left at this point -- rows where it
couldn't be recovered (blank or an ambiguous typo) were dropped earlier
rather than imputed. `Time` is imputed grouped by the now-complete `Day`
column, then `Smoker`, `Amount`, `Tip`, and `Partysize` group on the
complete `Day`+`Time` pair.

In [16]:
impute_categorical_grouped("Time", ["Day"])
impute_categorical_grouped("Smoker", ["Day", "Time"])

## 17. Impute the numeric columns

`Amount`, `Tip`, and `Partysize` are all right-skewed (mean > median in the
summary stats — e.g. `Amount` mean 21.21 vs. median 19.11, `Tip` mean 3.08
vs. median 2.85) because a handful of large parties/bills pull the mean
upward. We use the **median**, not the mean, for every numeric imputation
here because it represents the "typical" party in that `Day`+`Time` slot
without being dragged by those few outliers — using the mean would
systematically overstate the imputed values.

`Partysize` is rounded and cast back to `int` afterward since a party can't
have a fractional number of people.

In [17]:
impute_numeric_grouped("Amount", ["Day", "Time"])
impute_numeric_grouped("Tip", ["Day", "Time"])
impute_numeric_grouped("Partysize", ["Day", "Time"])
imputed["Partysize"] = imputed["Partysize"].round().astype(int)

## 18. Derived metric: `TipPct`

Tip as a percentage of the bill is the metric used throughout the later
Phase 3/4 analysis (it's a fairer comparison across bill sizes than the
raw tip amount).

In [18]:
imputed["TipPct"] = (imputed["Tip"] / imputed["Amount"]) * 100
imputed[["Amount", "Tip", "TipPct"]].head()

,Amount,Tip,TipPct
0,16.99,1.01,5.944673
1,10.34,1.66,16.054159
2,21.01,3.50,16.658734
3,23.68,3.31,13.978041
4,24.59,3.61,14.680765


## 19. Save the final cleaned + imputed dataset, and print the report

`restaurant_clean_final.csv` is this notebook's own output, kept as an
independent cross-check -- as of 2026-09-24, downstream EDA/visualization/
hypothesis testing reads `data/Restaurant_cleaned_final.csv` (the team's
agreed-final, teammate-provided dataset) instead. See `log.md`.

In [19]:
imputed.to_csv(OUT_FINAL, index=False)

print(f"Raw rows: {n_raw} | Final rows: {len(imputed)}")
print("\n--- Cleaning report ---")
for k, v in report.items():
    print(f"{k}: {v}")
print(f"\nSaved: {OUT_WITH_NA}")
print(f"Saved: {OUT_FINAL}")

Raw rows: 365 | Final rows: 352

--- Cleaning report ---
Amount: negative (invalid): 1
Tip: implausibly large (invalid): 3
Gender: typo values corrected: 21
Gender: unresolved -> missing: 0
Smoker: typo values corrected: 12
Smoker: missing/unresolved: 10
Day: typo values corrected: 14
Day: blank (dropped): 4
Day: ambiguous typo (dropped): 9
Time: typo values corrected: 10
Time: missing/unresolved: 5
Partysize: domain-invalid (<=0 or >6): 14
Partysize: originally blank: 3
Time: imputed via mode within Day group: 5
Smoker: imputed via mode within Day+Time group: 10
Amount: imputed via median within Day+Time group: 1
Tip: imputed via median within Day+Time group: 5
Partysize: imputed via median within Day+Time group: 17

Saved: ../data/restaurant_clean_with_na.csv
Saved: ../data/restaurant_clean_final.csv
